In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
import warnings
import streamlit as st
import glob
import os

# Code GPT 5.1 11-21-25
def rebase_time_gaps(df, time_col='time', gap_threshold=2.0, reset_gap=0.01):
    """
    Rebases timestamps when gaps exceed a threshold.
    
    Parameters:
        df : pd.DataFrame
            Must contain a numeric or datetime time column.
        time_col : str
            Name of the time column.
        gap_threshold : float
            Threshold (in seconds) for gap detection.
        reset_gap : float
            Gap to insert after rebasing (in seconds).
    Returns:
        pd.DataFrame with continuous time column.
    """
    df = df.copy()
    
    # Convert to numeric seconds if datetime
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        df[time_col] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()
    
    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)
    
    # Detect large gaps
    time_vals = df[time_col].to_numpy()
    gaps = np.diff(time_vals)
    
    # Keep track of total offset applied
    offset = 0.0
    adjusted_times = [time_vals[0]]
    
    for i, gap in enumerate(gaps, start=1):
        if gap > gap_threshold:
            # Increase offset by the size of the gap minus desired reset gap
            offset += (gap - reset_gap)
        adjusted_times.append(time_vals[i] - offset)
    
    df[time_col] = adjusted_times
    return df


def aursad_data():
    data_path = "../data/aursad"

    # Get all feather files, sorted in order (important for time series)
    feather_files = sorted(glob.glob(os.path.join(data_path, "part_*.feather")))

    # Load and concatenate
    start = 2
    stop = 6
    df_aursad = pd.concat([pd.read_feather(f) for f in feather_files[start:stop]], ignore_index=True)

    df_aursad = df_aursad.rename(columns={'timestamp': 'time'})
    df_aursad['time'] = df_aursad['time'] - df_aursad['time'].min()
    df_aursad = df_aursad.sort_values('time').reset_index(drop=True)
    df_aursad = rebase_time_gaps(df_aursad, time_col='time', gap_threshold=2.0, reset_gap=0.01)

    # Downsample
    df_aursad = df_aursad.iloc[::200]

    # Renaming to match CobotOps
    for i in range(6):
        df_aursad = df_aursad.rename(columns={f'actual_current_{i}': f'Current{i}'})
        df_aursad = df_aursad.rename(columns={f'actual_TCP_speed_{i}': f'Speed{i}'})
        df_aursad = df_aursad.rename(columns={f'joint_temperatures_{i}': f'Temperature{i}'})

    # simplify joint angle names
    for i in range(6):
        df_aursad = df_aursad.rename(columns={f'actual_q_{i}': f'q{i}'})

    # Encode labels for screwing failures
    df_aursad = pd.get_dummies(df_aursad, columns=['label'], prefix='label')
    label_names = ["Normal operation", "Damaged screw", "Extra assembly component", "Missing screw", "Damaged thread samples", "Screw Loosening"]

    for i, label in enumerate(label_names):
        df_aursad = df_aursad.rename(columns={f'label_{i}': label})
    df_aursad.head()

    return df_aursad

df = aursad_data()


In [5]:
df.to_feather('../data/aursad/aursad.feather')
df

,sample_nr,time,target_q_0,target_q_1,target_q_2,target_q_3,target_q_4,target_q_5,target_qd_0,target_qd_1,...,output_bit_register_66,output_bit_register_67,output_bit_register_70,output_bit_register_71,output_bit_register_72,Normal operation,Damaged screw,Extra assembly component,Missing screw,Screw Loosening
0,278,0.000,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.000000,0.000000,...,False,False,False,False,False,False,False,False,False,True
200,278,2.030,0.086604,-1.085948,1.304136,-0.174732,-0.037080,-1.605176,-0.007701,-0.000227,...,False,False,False,False,False,False,False,False,False,True
400,278,4.040,-1.437947,-1.130811,1.591008,-0.447633,-0.906855,-1.576459,-0.615464,-0.018111,...,False,False,False,False,False,False,False,False,False,True
600,278,6.050,-1.627345,-1.136384,1.626647,-0.481536,-1.014909,-1.572891,0.000000,0.000000,...,False,False,False,False,False,False,False,False,False,True
800,278,8.050,-1.627345,-1.136384,1.626647,-0.481536,-1.014909,-1.572891,0.000000,0.000000,...,False,False,True,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332400,130,3327.741,-0.182561,-2.289697,-1.151743,3.441964,0.802502,-1.572367,0.000000,0.000000,...,False,False,True,False,False,True,False,False,False,False
332600,130,3329.741,-0.182561,-2.289697,-1.151743,3.441964,0.802502,-1.572367,0.000000,0.000000,...,False,False,True,False,False,True,False,False,False,False
332800,130,3331.741,-0.182561,-2.289697,-1.151743,3.441964,0.802502,-1.572367,0.000000,0.000000,...,False,False,True,False,False,True,False,False,False,False
333000,130,3333.751,-0.120893,-2.014049,-0.589375,2.613780,0.610264,-1.579881,0.095832,0.428358,...,False,False,False,False,False,True,False,False,False,False


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1667 entries, 0 to 333200
Columns: 138 entries, sample_nr to Screw Loosening
dtypes: bool(12), float64(111), int32(1), int64(14)
memory usage: 1.6 MB


In [7]:
df.head()

,sample_nr,time,target_q_0,target_q_1,target_q_2,target_q_3,target_q_4,target_q_5,target_qd_0,target_qd_1,...,output_bit_register_66,output_bit_register_67,output_bit_register_70,output_bit_register_71,output_bit_register_72,Normal operation,Damaged screw,Extra assembly component,Missing screw,Screw Loosening
0,278,0.00,0.086743,-1.085944,1.304110,-0.174707,-0.037001,-1.605179,0.000000,0.000000,...,False,False,False,False,False,False,False,False,False,True
200,278,2.03,0.086604,-1.085948,1.304136,-0.174732,-0.037080,-1.605176,-0.007701,-0.000227,...,False,False,False,False,False,False,False,False,False,True
400,278,4.04,-1.437947,-1.130811,1.591008,-0.447633,-0.906855,-1.576459,-0.615464,-0.018111,...,False,False,False,False,False,False,False,False,False,True
600,278,6.05,-1.627345,-1.136384,1.626647,-0.481536,-1.014909,-1.572891,0.000000,0.000000,...,False,False,False,False,False,False,False,False,False,True
800,278,8.05,-1.627345,-1.136384,1.626647,-0.481536,-1.014909,-1.572891,0.000000,0.000000,...,False,False,True,False,False,False,False,False,False,True


In [8]:
import numpy as np
import pandas as pd

# Code GPT 5.1 11-21-25
def rebase_time_gaps(df, time_col='time', gap_threshold=2.0, reset_gap=0.01):
    """
    Rebases timestamps when gaps exceed a threshold.
    
    Parameters:
        df : pd.DataFrame
            Must contain a numeric or datetime time column.
        time_col : str
            Name of the time column.
        gap_threshold : float
            Threshold (in seconds) for gap detection.
        reset_gap : float
            Gap to insert after rebasing (in seconds).
    Returns:
        pd.DataFrame with continuous time column.
    """
    df = df.copy()
    
    # Convert to numeric seconds if datetime
    if not pd.api.types.is_numeric_dtype(df[time_col]):
        df[time_col] = pd.to_datetime(df[time_col])
        df[time_col] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()
    
    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)
    
    # Detect large gaps
    time_vals = df[time_col].to_numpy()
    gaps = np.diff(time_vals)
    
    # Keep track of total offset applied
    offset = 0.0
    adjusted_times = [time_vals[0]]
    
    for i, gap in enumerate(gaps, start=1):
        if gap > gap_threshold:
            # Increase offset by the size of the gap minus desired reset gap
            offset += (gap - reset_gap)
        adjusted_times.append(time_vals[i] - offset)
    
    df[time_col] = adjusted_times
    return df

df = rebase_time_gaps(df, time_col='time', gap_threshold=2.0, reset_gap=0.01)

In [9]:
def time_series_plots(df, error, feature_type):
    # Define column groups
    feature_type_lst = ["Current", "Speed", "Temperature"]
    unit_lst = ["A", "m/s", "Degrees C"]
    colors = px.colors.qualitative.Dark24

    unit = unit_lst[feature_type_lst.index(feature_type)]

    fig_lst = []

    # for feature_type, unit in zip(feature_type_lst, unit):
    cols1 = [f"{feature_type}_J{i}" for i in range(0, 3)]
    cols2 = [f"{feature_type}_J{i}" for i in range(3, 6)]

    # Create subplots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        subplot_titles=(f'{feature_type} Joints 0-2', f'{feature_type} Joints 3-5'),
                        vertical_spacing=0.1)

    # Add current traces to first subplot
    for i, col in enumerate(cols1):
        fig.add_trace(go.Scatter(x=df['time'], y=df[col],
                                name=col, mode='lines', line=dict(color=colors[i])), row=1, col=1)

    # Add speed traces to second subplot
    for i, col in enumerate(cols2):
        # Avoid black color for last joint
        if i != len(cols2) - 1:
            color = colors[i + 3] 
        else:
            color = colors[6]
        fig.add_trace(go.Scatter(x=df['time'], y=df[col], 
                                name=col, mode='lines', line=dict(color=color)), row=2, col=1)

    # Add yellow dots where grip_lost is True
    if error in df.columns:
        df_flag = df[df[error] == True]
        
        if not df_flag.empty:
            # Add markers to subplot 1 (for each joint in cols1)
            for i, col in enumerate(cols1):
                fig.add_trace(go.Scatter(
                    x=df_flag['time'], 
                    y=df_flag[col],
                    mode='markers',
                    marker=dict(color='yellow', size=6, symbol='circle'),
                    name=error,
                    showlegend=(i == 0)  # Only show legend for first occurrence
                ), row=1, col=1)
            
            # Add markers to subplot 2 (for each joint in cols2)
            for i, col in enumerate(cols2):
                fig.add_trace(go.Scatter(
                    x=df_flag['time'], 
                    y=df_flag[col],
                    mode='markers',
                    marker=dict(color='yellow', size=6, symbol='circle'),
                    name=error,
                    showlegend=False  
                ), row=2, col=1)

    # Add rangeslider to bottom subplot only
    fig.update_xaxes(rangeslider_visible=True, row=2, col=1)

    fig.update_xaxes(title_text="Time (s)", row=2, col=1, rangeslider_visible=True)
    fig.update_yaxes(title_text=f"{feature_type} ({unit})", row=1, col=1)
    fig.update_yaxes(title_text=f"{feature_type} ({unit})", row=2, col=1)

    # Update layout
    fig.update_layout(height=800)
    fig_lst.append(fig)

    return fig_lst

error_lst = ["Damaged screw", "Extra assembly component", "Missing screw"]

fig_lst = time_series_plots(df, error_lst[1], "Current")
for fig in fig_lst:
    fig.show()

KeyError: 'Current_J0'